In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


учёные взяли неизвестную молекулу, разбили её на фрагменты с помощью масс-спектрометра и получили список чисел.

Наша задача — по этим числам угадать, как выглядит исходная молекула.

In [2]:
import os
import sys
import importlib.util
import subprocess
from pathlib import Path
import numpy as np 
import pandas as pd 

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print("Python:", sys.version)

# Проверка GPU через PyTorch
if importlib.util.find_spec("torch"):
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PyTorch: not installed")

# Проверка библиотек
print("\nLIBRARIES")
for package in [
    "pandas", "pyarrow", "numpy", "scipy",
    "rdkit", "lightgbm", "sklearn"
]:
    print(f"{package}:",
          "OK" if importlib.util.find_spec(package) else "MISSING")

# Поиск входных файлов Kaggle
print("\nINPUT FILES")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith((".parquet", ".csv", ".smi", ".sdf")):
            print(os.path.join(root, f))

print("\nOUTPUT DIRECTORY EXISTS:",
      os.path.isdir("/kaggle/working"))

ENVIRONMENT
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

LIBRARIES
pandas: OK
pyarrow: OK
numpy: OK
scipy: OK
rdkit: MISSING
lightgbm: OK
sklearn: OK

INPUT FILES
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet

OUTPUT DIRECTORY EXISTS: True


In [3]:
def find_file(name, path=Path("/kaggle/input")):
    for root, dirs, files in os.walk(path):
        if name in files:
            return os.path.join(root, name)
    raise FileNotFoundError(f"Файл {name} не найден в директории {path}")

ROOT = Path("/kaggle/input")

TRAIN_PATH = find_file("train.parquet")
TEST_PATH = find_file("test.parquet")
SAMPLE_PATH = find_file("sample_submission.csv")

print("Train path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
print("Sample path:", SAMPLE_PATH)

# 3. Загрузка данных
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest preview:")
display(test.head())

print("\nSample submission:")
display(sample.head())


Train path: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
Test path: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet
Sample path: /kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv

Train shape: (2539608, 18)
Test shape: (1213, 12)
Sample shape: (400, 2)

Train columns:
['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'ms2_mzs', 'ms2_normalized_intensities', 'num_peaks', 'base_peak_intensity', 'collision_energy_ev', 'collision_energy_orig', 'collision_energy_orig_units']

Test preview:


,molecule_id,spectrum_id,ms2_mzs,ms2_normalized_intensities,base_peak_intensity,adduct,ionization_mode,instrument_type,precursor_mz,collision_energy_orig,collision_energy_ev,collision_energy_orig_units
0,m_005e53,s_4d19d522,"[67.0414, 73.0074, 74.0142, 74.0174, 75.0238, ...","[0.0016501163269310492, 0.0004909040749095276,...",2711324.0,[M+H]+,positive,timsTOF,375.0709,"20,40,60","[20.0, 40.0, 60.0]",eV
1,m_005e53,s_5b6e28ff,"[67.0414, 73.0074, 74.0174, 75.0236, 76.0311, ...","[0.0029581573469943693, 0.0008800418929033316,...",1512428.0,[M+H]+,positive,timsTOF,375.0709,40,[40.0],eV
2,m_005e53,s_6e514292,"[74.0142, 75.0238, 76.0313, 77.0357, 77.0398, ...","[0.0071407135863519105, 0.009720343101840298, ...",989677.0,[M+H]+,positive,timsTOF,375.0709,60,[60.0],eV
3,m_005e53,s_855ae19c,"[42.9863, 45.6701, 69.5715, 78.9191, 78.929, 7...","[0.0014297061159650516, 0.0014297061159650516,...",6295.0,[M-H]-,negative,timsTOF,373.0562,-60,[60.0],eV
4,m_006153,s_3c74e9fe,"[65.0391, 77.0385, 77.1154, 78.0479, 85.0515, ...","[0.000254435863675019, 0.0009782620275780901, ...",3191374.0,[M+H]+,positive,timsTOF,342.1569,20,[20.0],eV



Sample submission:


,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3,m_01e922,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4,m_0259d4,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...




train: 2 539 608 строк - Более 2,5 млн известных спектров для обучения и поиска

test: 1 213 строк - Спектры неизвестных молекул, которые нужно определить

sample: 400 строк - Пример файла, который мы должны сдать

400 molecules - Всего 400 неизвестных молекул, каждая может иметь несколько спектров





1 213 спектров, но 400 молекул,  так как одну молекулу измеряют несколько раз, например при разных энергиях столкновения. Мы должны объединить эти измерения и выдать один список предсказаний на молекулу.

molecule_id -- Идентификатор неизвестной молекулы

spectrum_id -- Идентификатор конкретного измерения

precursor_mz -- Масса-заряд исходного иона до фрагментации

adduct -- Заряженная форма молекулы, которую измеряет прибор

ms2_mzs -- Массы фрагментов

ms2_normalized_intensities -- Интенсивности соответствующих фрагментов

normalized_smiles -- Строковое описание известной химической структуры

inchikey14 -- Ключ, позволяющий объединять одинаковые структуры по связности атомов

In [4]:
from collections import Counter

print("TEST adducts:")
print(test["adduct"].value_counts(dropna=False))


print("\nTRAIN adducts:")
print(train["adduct"].value_counts(dropna=False).head(20))

print("\nTest instrument types:")
print(test["instrument_type"].value_counts(dropna=False))

print("\nTest molecules:", test["molecule_id"].nunique())
print("Spectra per molecule:")
print(test.groupby("molecule_id").size().describe())

print("\nPrecursor m/z range:")
print(test["precursor_mz"].describe())

TEST adducts:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64

TRAIN adducts:
adduct
[M+H]+           1324585
[M-H]-            465688
[2M+Na]+          190033
[M+Na]+           178092
[2M+H]+            92041
[M+CH2O2-H]-       83188
[2M-H]-            56172
[M+NH4]+           53452
[M+Cl]-            27511
[M-H2O+H]+         21682
[M+K]+             18955
[M]+                8818
[M+C2H4O2-H]-       7436
[M-2H2O+H]+         3439
[M+2H]2+            1717
[2M+Na-2H]-         1431
[2M+CH2O2-H]-        665
[M-H2O]+             624
[M-CH3]-             619
[M-2H]-              454
Name: count, dtype: int64

Test instrument types:
instrument_type
timsTOF    1213
Name: count, dtype: int64

Test molecules: 400
Spectra per molecule:
count    400.000000
mean       3.032500
std        1.382469
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000

Шаг 1. Получаем спектр

Например, набор пиков с массами и интенсивностями. Это своего рода химический отпечаток молекулы.

In [5]:
ADDUCT_DELTA = {
    "[M+H]+": 1.0072764666,
    "[M+NH4]+": 18.033823,
    "[M-H2O+H]+": -17.003289,
    "[M-2H2O+H]+": -35.013854,
    "[M+Na]+": 22.989218,
    "[M+K]+": 38.963158,
    "[M-H]-": -1.0072764666,
    "[M+CH2O2-H]-": 44.998203,
    "[M+Cl]-": 34.969402,
}

def neutral_mass(mz, adduct):
    delta = ADDUCT_DELTA.get(adduct)
    if delta is None or pd.isna(mz):
        return np.nan
    return float(mz) - delta

ref = train.copy()

ref["neutral_mass"] = [
    neutral_mass(mz, adduct)
    for mz, adduct in zip(ref["precursor_mz"], ref["adduct"])
]

ref = ref.dropna(subset=["neutral_mass", "normalized_smiles"]).copy()
ref = ref[np.isfinite(ref["neutral_mass"])].copy()
ref = ref.sort_values("neutral_mass").reset_index(drop=True)

ref_mass = ref["neutral_mass"].to_numpy(dtype=np.float64)

print("Reference spectra retained:", len(ref))
print("Unique structures:", ref["inchikey14"].nunique())
print("Mass range:", ref_mass.min(), "-", ref_mass.max())
print("Unsupported adduct spectra excluded:", len(train) - len(ref))

Reference spectra retained: 2176592
Unique structures: 269843
Mass range: 31.0422235334 - 2430334.0072764666
Unsupported adduct spectra excluded: 363016


Что такое аддукты и почему они нас запутали?
Это самая техническая часть, но её легко понять на примере.

Масс-спектрометр измеряет не всегда массу самой молекулы, а массу заряженной формы этой молекулы.

Например:

[M+H]+ — молекула присоединила протон.

[M-H]- — молекула потеряла протон.

[M+Na]+ — молекула присоединила натрий.

[2M+H]+ — две молекулы объединились и получили протон.

Поэтому мы написали словарь ADDUCT_DELTA, который помогает пересчитать измеренную массу в приблизительную массу исходной молекулы.

Например, если прибор измерил [M+H]+ с массой 301,123, мы вычитаем массу протона и получаем примерно 300,116 Да.

Это важно, потому что иначе мы будем искать в базе молекулы с неправильной массой.

In [6]:
print("Аддукты в test:")
print(test["adduct"].value_counts(dropna=False))

print("\nАддукты в train:")
print(train["adduct"].value_counts(dropna=False))

print("\nАддукты test, отсутствующие в словаре:")
print(sorted(set(test["adduct"].dropna()) - set(ADDUCT_DELTA)))

print("\nАддукты train, отсутствующие в словаре:")
print(sorted(set(train["adduct"].dropna()) - set(ADDUCT_DELTA)))

Аддукты в test:
adduct
[M+H]+          959
[M-H]-          193
[M+CH2O2-H]-     31
[M+Na]+          22
[M+NH4]+          4
[M+K]+            2
[M+Cl]-           2
Name: count, dtype: int64

Аддукты в train:
adduct
[M+H]+         1324585
[M-H]-          465688
[2M+Na]+        190033
[M+Na]+         178092
[2M+H]+          92041
                ...   
[M-H2O2]2+           1
[2M-H5O3]+           1
[M+C2H7OS]+          1
[M+HO]+              1
[3M+K]+              1
Name: count, Length: 121, dtype: int64

Аддукты test, отсутствующие в словаре:
[]

Аддукты train, отсутствующие в словаре:
['[2M+C2H4N]+', '[2M+C2H4O2-H]-', '[2M+CH2O2-H]-', '[2M+Ca-H]+', '[2M+Ca]2+', '[2M+H-O2]+', '[2M+H]+', '[2M+K]+', '[2M+NH4]+', '[2M+Na-2H]-', '[2M+Na-H2O]+', '[2M+Na-H5N]-', '[2M+Na-O2]+', '[2M+Na-O3]+', '[2M+Na2-H3]+', '[2M+Na2-H]+', '[2M+Na3-H4]-', '[2M+Na]+', '[2M-H3O2]+', '[2M-H3]-', '[2M-H5O3]+', '[2M-HO]+', '[2M-H]+', '[2M-H]-', '[3M+Ca-H]+', '[3M+Ca]2+', '[3M+H4N]+', '[3M+K]+', '[3M+Na-H4O2]+', '[3M+

In [7]:

import numpy as np

FRAGMENT_TOL_DA = 0.03

def clean_spectrum(mzs, intensities, precursor_mz=None,
                   min_intensity=0.0):
    """Возвращает отсортированные массы и нормированные интенсивности."""
    if mzs is None or intensities is None:
        return np.array([]), np.array([])

    mzs = np.asarray(mzs, dtype=np.float64).ravel()
    intensities = np.asarray(intensities, dtype=np.float64).ravel()

    if len(mzs) != len(intensities):
        return np.array([]), np.array([])

    valid = (
        np.isfinite(mzs) & np.isfinite(intensities)
        & (mzs > 0) & (intensities > min_intensity)
    )
    mzs, intensities = mzs[valid], intensities[valid]

    if len(mzs) == 0 or intensities.max() <= 0:
        return np.array([]), np.array([])

    # Нормализация до максимальной интенсивности 1.
    intensities = intensities / intensities.max()

    order = np.argsort(mzs)
    return mzs[order], intensities[order]


def spectral_cosine(mz1, int1, mz2, int2,
                    tol=FRAGMENT_TOL_DA):
    """Косинусное сходство с попарным сопоставлением пиков."""
    mz1 = np.asarray(mz1, dtype=float)
    int1 = np.asarray(int1, dtype=float)
    mz2 = np.asarray(mz2, dtype=float)
    int2 = np.asarray(int2, dtype=float)

    if len(mz1) == 0 or len(mz2) == 0:
        return 0.0

    norm1 = np.linalg.norm(int1)
    norm2 = np.linalg.norm(int2)
    if norm1 == 0 or norm2 == 0:
        return 0.0

    int1 = int1 / norm1
    int2 = int2 / norm2

    # Для каждого пика первого спектра выбираем
    # ближайший пик второго спектра в пределах допуска.
    order2 = np.argsort(mz2)
    mz2, int2 = mz2[order2], int2[order2]

    dot = 0.0
    for mz, intensity in zip(mz1, int1):
        left = np.searchsorted(mz2, mz - tol, side="left")
        right = np.searchsorted(mz2, mz + tol, side="right")

        if left == right:
            continue

        j = left + np.argmin(np.abs(mz2[left:right] - mz))
        dot += intensity * int2[j]

    return float(np.clip(dot, 0.0, 1.0))


Код выше - очищаем пики, нормализуем интенсивности и вычисляем косинусное сходство с сопоставлением пиков в пределах заданного допуска.

Что такое аддукты?

Масс-спектрометр измеряет не всегда массу самой молекулы, а массу заряженной формы этой молекулы.

Например:

[M+H]+ — молекула присоединила протон.

[M-H]- — молекула потеряла протон.

[M+Na]+ — молекула присоединила натрий.

[2M+H]+ — две молекулы объединились и получили протон.

Поэтому мы написали словарь ADDUCT_DELTA, который помогает пересчитать измеренную массу в приблизительную массу исходной молекулы.

Например, если прибор измерил [M+H]+ с массой 301,123, мы вычитаем массу протона и получаем примерно 300,116 Да.

Это важно, потому что иначе мы будем искать в базе молекулы с неправильной массой.

Шаг 2. Ищем похожие известные молекулы

Сравниваем неизвестный спектр с миллионами спектров, для которых структура уже известна.

проверяем, сколько кандидатов найдётся
Здесь мы пока не сравниваем спектры. Только проверяем, сколько обучающих записей имеют массу, близкую к массе неизвестной молекулы.

Допуск 20 ppm — это небольшой относительный интервал вокруг измеренной массы.

In [8]:
MASS_TOL_PPM = 20.0

def find_mass_candidates(mz, adduct):
    mass = neutral_mass(mz, adduct)

    if not np.isfinite(mass):
        return ref.iloc[0:0]

    tol = mass * MASS_TOL_PPM * 1e-6

    left = np.searchsorted(ref_mass, mass - tol, side="left")
    right = np.searchsorted(ref_mass, mass + tol, side="right")

    return ref.iloc[left:right]

# Проверим первые пять тестовых спектров.
for idx, row in test.head(5).iterrows():
    candidates = find_mass_candidates(
        row["precursor_mz"],
        row["adduct"]
    )

    print(
        f"\nspectres {idx}: {row['molecule_id']}, "
        f"adduct={row['adduct']}, "
        f"candidates={len(candidates)}, "
        f"unique structures={candidates['inchikey14'].nunique()}"
    )

    display(
        candidates[
            ["normalized_smiles", "molecular_formula",
             "neutral_mass", "inchikey14"]
        ].head(3)
    )


spectres 0: m_005e53, adduct=[M+H]+, candidates=285, unique structures=39


,normalized_smiles,molecular_formula,neutral_mass,inchikey14
1513177,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513178,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513179,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG



spectres 1: m_005e53, adduct=[M+H]+, candidates=285, unique structures=39


,normalized_smiles,molecular_formula,neutral_mass,inchikey14
1513177,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513178,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513179,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG



spectres 2: m_005e53, adduct=[M+H]+, candidates=285, unique structures=39


,normalized_smiles,molecular_formula,neutral_mass,inchikey14
1513177,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513178,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG
1513179,O=C(c1cc2cccc(C(F)(F)F)c2[nH]c1=O)N1CCS(=O)(=O...,C15H13F3N2O4S,374.056224,RSXKVWLUBRSVDG



spectres 3: m_005e53, adduct=[M-H]-, candidates=277, unique structures=39


,normalized_smiles,molecular_formula,neutral_mass,inchikey14
1513173,COc1ccc2c(c1)N(S(=O)(=O)c1ccc(C(F)(F)F)nc1)CCO2,C15H13F3N2O4S,374.056124,UOCUUZXFTYVPJJ
1513174,COc1ccc2c(c1)N(S(=O)(=O)c1ccc(C(F)(F)F)nc1)CCO2,C15H13F3N2O4S,374.056124,UOCUUZXFTYVPJJ
1513175,COc1ccc2c(c1)N(S(=O)(=O)c1ccc(C(F)(F)F)nc1)CCO2,C15H13F3N2O4S,374.056124,UOCUUZXFTYVPJJ



spectres 4: m_006153, adduct=[M+H]+, candidates=1580, unique structures=262


,normalized_smiles,molecular_formula,neutral_mass,inchikey14
1173648,CC1(c2ccccc2)CCCN(C(=O)c2ccc(C(=O)O)cc2F)C1,C20H20FNO3,341.142876,IXQSNDSOTJDBBP
1173649,O=C(O)Cc1cccc(C(=O)NC2(Cc3cccc(F)c3)CCC2)c1,C20H20FNO3,341.142876,MMJPQLPZXQRWBI
1173650,CC1(c2ccccc2)CCCN(C(=O)c2ccc(C(=O)O)cc2F)C1,C20H20FNO3,341.142876,IXQSNDSOTJDBBP


**Следующий шаг — спектральное сходство**

У нас есть два источника информации:

Масса молекулы: помогает найти возможные структуры.

Фрагментные пики: помогают понять, какая из этих структур лучше объясняет экспериментальный спектр.

Например, если у неизвестной молекулы есть пики около 74.0142, 75.0238 и 76.0313, мы будем искать кандидатов, у которых соответствующие фрагменты также присутствуют.

Для начала используем косинусное сходство между спектрами. Это распространённый базовый метод сравнения масс-спектров.

In [9]:
MASS_TOL_PPM = 20.0
FRAGMENT_TOL_DA = 0.03

def rank_molecule(molecule_id, top_n=25):
    group = test[test["molecule_id"] == molecule_id]

    if group.empty:
        raise ValueError(f"Unknown molecule_id: {molecule_id}")

    # Накопленные оценки структуры по отдельным спектрам.
    accumulated = {}

    for _, query in group.iterrows():
        qmass = neutral_mass(
            query["precursor_mz"], query["adduct"]
        )

        if not np.isfinite(qmass):
            continue

        tol = qmass * MASS_TOL_PPM * 1e-6
        left = np.searchsorted(ref_mass, qmass - tol)
        right = np.searchsorted(ref_mass, qmass + tol)

        candidates = ref.iloc[left:right]

        if candidates.empty:
            continue

        q_mz, q_int = clean_spectrum(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            query["precursor_mz"],
        )

        # Для каждой структуры сохраняем лучшее сходство
        # среди всех её эталонных спектров.
        spectrum_scores = {}

        for _, candidate in candidates.iterrows():
            key = candidate["inchikey14"]

            r_mz, r_int = clean_spectrum(
                candidate["ms2_mzs"],
                candidate["ms2_normalized_intensities"],
                candidate["precursor_mz"],
            )

            sim = spectral_cosine(
                q_mz, q_int, r_mz, r_int
            )

            if key not in spectrum_scores or sim > spectrum_scores[key][0]:
                spectrum_scores[key] = (
                    sim,
                    candidate["normalized_smiles"],
                    candidate["molecular_formula"],
                    candidate["neutral_mass"],
                )

        for key, values in spectrum_scores.items():
            sim, smiles, formula, mass = values

            if key not in accumulated:
                accumulated[key] = {
                    "smiles": smiles,
                    "formula": formula,
                    "neutral_mass": mass,
                    "scores": [],
                }

            accumulated[key]["scores"].append(sim)

    results = []

    for key, values in accumulated.items():
        scores = values["scores"]

        results.append({
            "inchikey14": key,
            "smiles": values["smiles"],
            "formula": values["formula"],
            "neutral_mass": values["neutral_mass"],
            "mean_similarity": float(np.mean(scores)),
            "best_similarity": float(np.max(scores)),
            "spectra_matched": len(scores),
        })

    result = pd.DataFrame(results)

    if result.empty:
        return result

    # Основной рейтинг — среднее сходство по спектрам,
    # для которых структура прошла фильтр массы.
    return (
        result.sort_values(
            ["mean_similarity", "best_similarity"],
            ascending=False,
        )
        .head(top_n)
        .reset_index(drop=True)
    )


ranked_all = rank_molecule("m_005e53", top_n=25)
display(ranked_all)

,inchikey14,smiles,formula,neutral_mass,mean_similarity,best_similarity,spectra_matched
0,QCTQNXJUUXYSRK,CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1,C18H19BrN2O2,374.063624,1.000000,1.000000,4
1,XZGCKUAAHQVVHY,O=C(O)c1cccnc1CN1CCC(c2ccccc2Br)CC1,C18H19BrN2O2,374.063924,0.730519,0.998417,4
2,SCIHCHFEKPRVSD,CC1CC(c2ccc(Br)cc2)CN1Cc1ncccc1C(=O)O,C18H19BrN2O2,374.064124,0.551711,1.000000,4
3,BWNLJQYJJFXPQO,Cc1ccc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)nc1,C18H19BrN2O2,374.063624,0.345560,0.999026,4
4,YYPOJIOVHJAXPU,Cc1ncccc1CN1CC(C(=O)O)C(c2ccc(Br)cc2)C1,C18H19BrN2O2,374.063524,0.341947,1.000000,4
5,FZOHNMLMGAZTTJ,Cc1ccc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)cn1,C18H19BrN2O2,374.064024,0.338463,1.000000,4
6,HATBONOIGREPOF,Cc1cncc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)c1,C18H19BrN2O2,374.063824,0.330989,0.998766,4
7,MTOOHBPCPVRUHN,Cc1ccc(C(=O)NCC(=O)Nc2ccc(Br)c(C)c2)cc1C,C18H19BrN2O2,374.062976,0.254526,1.000000,4
8,ZSDDGNZHLINBRL,Cc1nc(C2CC2)oc1C(=O)N1CCCc2c(Br)cccc2C1,C18H19BrN2O2,374.063324,0.199931,0.494517,4
9,MIIXAPDCACTUNC,Cc1cc(C)c(NC(=O)c2ccn(-c3ccc(Cl)c(Cl)c3)n2)c(C)n1,C18H16Cl2N4O,374.071024,0.150753,0.449075,3


In [10]:
molecule_id = "m_005e53"

print("Спектры молекулы:")
display(
    test.loc[
        test["molecule_id"] == molecule_id,
        [
            "spectrum_id",
            "adduct",
            "precursor_mz",
            "collision_energy_ev",
        ],
    ]
)

print("\nТоп кандидатов:")
display(
    ranked_all[
        [
            "smiles",
            "formula",
            "mean_similarity",
            "best_similarity",
            "spectra_matched",
        ]
    ].head(10)
)

Спектры молекулы:


,spectrum_id,adduct,precursor_mz,collision_energy_ev
0,s_4d19d522,[M+H]+,375.0709,"[20.0, 40.0, 60.0]"
1,s_5b6e28ff,[M+H]+,375.0709,[40.0]
2,s_6e514292,[M+H]+,375.0709,[60.0]
3,s_855ae19c,[M-H]-,373.0562,[60.0]



Топ кандидатов:


,smiles,formula,mean_similarity,best_similarity,spectra_matched
0,CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1,C18H19BrN2O2,1.000000,1.000000,4
1,O=C(O)c1cccnc1CN1CCC(c2ccccc2Br)CC1,C18H19BrN2O2,0.730519,0.998417,4
2,CC1CC(c2ccc(Br)cc2)CN1Cc1ncccc1C(=O)O,C18H19BrN2O2,0.551711,1.000000,4
3,Cc1ccc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)nc1,C18H19BrN2O2,0.345560,0.999026,4
4,Cc1ncccc1CN1CC(C(=O)O)C(c2ccc(Br)cc2)C1,C18H19BrN2O2,0.341947,1.000000,4
5,Cc1ccc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)cn1,C18H19BrN2O2,0.338463,1.000000,4
6,Cc1cncc(CN2CC(C(=O)O)C(c3ccc(Br)cc3)C2)c1,C18H19BrN2O2,0.330989,0.998766,4
7,Cc1ccc(C(=O)NCC(=O)Nc2ccc(Br)c(C)c2)cc1C,C18H19BrN2O2,0.254526,1.000000,4
8,Cc1nc(C2CC2)oc1C(=O)N1CCCc2c(Br)cccc2C1,C18H19BrN2O2,0.199931,0.494517,4
9,Cc1cc(C)c(NC(=O)c2ccn(-c3ccc(Cl)c(Cl)c3)n2)c(C)n1,C18H16Cl2N4O,0.150753,0.449075,3


проверяем, не является ли это дубликатом


In [11]:
# Берём первого кандидата из твоего рейтинга.
best = ranked_all.iloc[0]
best_key = best["inchikey14"]

print("Best candidate InChIKey14:", best_key)
print("SMILES:", best["smiles"])

# Находим все обучающие записи этой структуры.
matches = ref[ref["inchikey14"] == best_key].copy()

print("\nОбучающих спектров этой структуры:", len(matches))
print("\nИсточники:")
if "ingest_lib" in matches.columns:
    display(matches["ingest_lib"].value_counts().head(15))

# Сравниваем тестовые спектры с лучшими эталонными спектрами
# именно этой структуры.
check_rows = []

for _, query in test[test["molecule_id"] == "m_005e53"].iterrows():
    q_mz, q_int = clean_spectrum(
        query["ms2_mzs"],
        query["ms2_normalized_intensities"],
        query["precursor_mz"],
    )

    best_score = -1
    best_ref = None

    for _, candidate in matches.iterrows():
        r_mz, r_int = clean_spectrum(
            candidate["ms2_mzs"],
            candidate["ms2_normalized_intensities"],
            candidate["precursor_mz"],
        )

        score = spectral_cosine(q_mz, q_int, r_mz, r_int)

        if score > best_score:
            best_score = score
            best_ref = candidate

    check_rows.append({
        "spectrum_id": query["spectrum_id"],
        "test_adduct": query["adduct"],
        "test_energy": str(query["collision_energy_ev"]),
        "best_similarity": best_score,
        "reference_library": (
            best_ref["ingest_lib"] if best_ref is not None else None
        ),
        "reference_adduct": (
            best_ref["adduct"] if best_ref is not None else None
        ),
        "reference_energy": (
            best_ref["collision_energy_orig"] if best_ref is not None else None
        ),
        "reference_precursor_mz": (
            best_ref["precursor_mz"] if best_ref is not None else np.nan
        ),
    })

display(pd.DataFrame(check_rows))

Best candidate InChIKey14: QCTQNXJUUXYSRK
SMILES: CN(Cc1ncccc1C(=O)O)CC1(c2ccc(Br)cc2)CC1

Обучающих спектров этой структуры: 5

Источники:


ingest_lib
enveda-180    5
Name: count, dtype: int64

,spectrum_id,test_adduct,test_energy,best_similarity,reference_library,reference_adduct,reference_energy,reference_precursor_mz
0,s_4d19d522,[M+H]+,[20. 40. 60.],1.0,enveda-180,[M+H]+,"20,40,60",375.0709
1,s_5b6e28ff,[M+H]+,[40.],1.0,enveda-180,[M+H]+,40,375.0709
2,s_6e514292,[M+H]+,[60.],1.0,enveda-180,[M+H]+,60,375.0709
3,s_855ae19c,[M-H]-,[60.],1.0,enveda-180,[M-H]-,-60,373.0562


Cледующий код проверит, для скольких тестовых молекул в enveda-180 есть структура с совпадающей массой. Это ещё не поиск точных спектральных совпадений, но полезная оценка покрытия.

In [12]:
import numpy as np
import pandas as pd

MASS_TOL_PPM = 20.0

# Отдельный справочник спектров с того же прибора.
ref_enveda = ref[
    ref["ingest_lib"].eq("enveda-180")
].copy()

ref_enveda = ref_enveda.sort_values("neutral_mass").reset_index(drop=True)
enveda_mass = ref_enveda["neutral_mass"].to_numpy(dtype=np.float64)

def enveda_mass_candidates(mz, adduct):
    mass = neutral_mass(mz, adduct)

    if not np.isfinite(mass):
        return 0

    tol = mass * MASS_TOL_PPM * 1e-6

    left = np.searchsorted(enveda_mass, mass - tol, side="left")
    right = np.searchsorted(enveda_mass, mass + tol, side="right")

    return right - left

test["enveda_mass_candidates"] = [
    enveda_mass_candidates(row["precursor_mz"], row["adduct"])
    for _, row in test.iterrows()
]

coverage = (
    test.groupby("molecule_id")["enveda_mass_candidates"]
    .max()
)

print("Всего молекул:", len(coverage))
print("Молекул с кандидатами в enveda-180:", (coverage > 0).sum())
print("Молекул без кандидатов:", (coverage == 0).sum())

print("\nРаспределение числа кандидатов на спектр:")
print(test["enveda_mass_candidates"].describe())

display(
    test[[
        "molecule_id", "spectrum_id", "adduct",
        "precursor_mz", "enveda_mass_candidates"
    ]].head(15)
)

Всего молекул: 400
Молекул с кандидатами в enveda-180: 400
Молекул без кандидатов: 0

Распределение числа кандидатов на спектр:
count    1213.000000
mean      618.904369
std       476.727561
min         4.000000
25%       238.000000
50%       483.000000
75%       937.000000
max      1988.000000
Name: enveda_mass_candidates, dtype: float64


,molecule_id,spectrum_id,adduct,precursor_mz,enveda_mass_candidates
0,m_005e53,s_4d19d522,[M+H]+,375.0709,133
1,m_005e53,s_5b6e28ff,[M+H]+,375.0709,133
2,m_005e53,s_6e514292,[M+H]+,375.0709,133
3,m_005e53,s_855ae19c,[M-H]-,373.0562,134
4,m_006153,s_3c74e9fe,[M+H]+,342.1569,1102
5,m_006153,s_54245904,[M-H]-,340.1416,1098
6,m_006153,s_59a80abf,[M-H]-,340.1416,1098
7,m_006153,s_7151fa86,[M+H]+,342.1569,1102
8,m_006153,s_86472581,[M-H]-,340.1416,1098
9,m_006153,s_cb15f0ac,[M+H]+,342.1568,1106


итоговый алгоритм будет работать в три этапа:

Точное спектральное совпадение: искать соответствия в enveda-180 и других библиотеках.

Спектральное сходство: если точного совпадения нет, ранжировать структуры с подходящей массой по сходству фрагментов.

Резервные кандидаты: если библиотека не даёт хороших совпадений, использовать более широкий поиск по обучающим структурам.

Шаг 3. Предсказываем структуру

Выбираем до 25 наиболее вероятных молекул и записываем их в формате SMILES.

In [13]:
from collections import defaultdict

# --------------------------------------------------
# 1. Подготовка справочника
# --------------------------------------------------

MASS_TOL_PPM = 20.0
FRAGMENT_TOL_DA = 0.03
TOP_N = 25

ref_enveda = ref[
    ref["ingest_lib"].eq("enveda-180")
].copy()

ref_enveda = (
    ref_enveda
    .dropna(subset=["neutral_mass", "normalized_smiles", "inchikey14"])
    .sort_values("neutral_mass")
    .reset_index(drop=True)
)

enveda_mass = ref_enveda["neutral_mass"].to_numpy(dtype=np.float64)

print("Эталонных спектров:", len(ref_enveda))
print("Уникальных структур:", ref_enveda["inchikey14"].nunique())


# --------------------------------------------------
# 2. Ранжирование кандидатов для одной молекулы
# --------------------------------------------------

def rank_molecule_final(molecule_id, top_n=TOP_N):
    group = test[test["molecule_id"] == molecule_id]

    # Словарь: структура -> оценки по спектрам
    scores_by_structure = defaultdict(list)
    smiles_by_structure = {}
    formula_by_structure = {}

    for _, query in group.iterrows():
        qmass = neutral_mass(
            query["precursor_mz"], query["adduct"]
        )

        if not np.isfinite(qmass):
            continue

        tol = qmass * MASS_TOL_PPM * 1e-6

        left = np.searchsorted(
            enveda_mass, qmass - tol, side="left"
        )
        right = np.searchsorted(
            enveda_mass, qmass + tol, side="right"
        )

        candidates = ref_enveda.iloc[left:right]

        if candidates.empty:
            continue

        # Сначала предпочитаем тот же аддукт.
        same_adduct = candidates[
            candidates["adduct"] == query["adduct"]
        ]

        if not same_adduct.empty:
            candidates = same_adduct

        q_mz, q_int = clean_spectrum(
            query["ms2_mzs"],
            query["ms2_normalized_intensities"],
            query["precursor_mz"],
        )

        # Для каждой структуры оставляем лучший спектральный
        # результат среди её записей в текущем пуле кандидатов.
        per_query = {}

        for _, candidate in candidates.iterrows():
            key = candidate["inchikey14"]

            r_mz, r_int = clean_spectrum(
                candidate["ms2_mzs"],
                candidate["ms2_normalized_intensities"],
                candidate["precursor_mz"],
            )

            sim = spectral_cosine(
                q_mz, q_int, r_mz, r_int,
                tol=FRAGMENT_TOL_DA,
            )

            if key not in per_query or sim > per_query[key][0]:
                per_query[key] = (
                    sim,
                    candidate["normalized_smiles"],
                    candidate["molecular_formula"],
                )

        for key, (sim, smiles, formula) in per_query.items():
            scores_by_structure[key].append(float(sim))
            smiles_by_structure[key] = smiles
            formula_by_structure[key] = formula

    # Сводим оценки нескольких спектров в один рейтинг.
    results = []

    for key, scores in scores_by_structure.items():
        results.append({
            "inchikey14": key,
            "smiles": smiles_by_structure[key],
            "formula": formula_by_structure[key],
            "mean_similarity": float(np.mean(scores)),
            "best_similarity": float(np.max(scores)),
            "n_spectra_scored": len(scores),
        })

    if not results:
        return pd.DataFrame(
            columns=[
                "inchikey14", "smiles", "formula",
                "mean_similarity", "best_similarity",
                "n_spectra_scored",
            ]
        )

    result = pd.DataFrame(results)

    return (
        result.sort_values(
            ["mean_similarity", "best_similarity"],
            ascending=False,
        )
        .drop_duplicates("inchikey14")
        .head(top_n)
        .reset_index(drop=True)
    )


# --------------------------------------------------
# 3. Обрабатываем все молекулы
# --------------------------------------------------

molecule_ids = test["molecule_id"].drop_duplicates().tolist()
rankings = {}

for i, molecule_id in enumerate(molecule_ids, start=1):
    rankings[molecule_id] = rank_molecule_final(molecule_id)

    if i % 25 == 0 or i == len(molecule_ids):
        print(f"Обработано {i}/{len(molecule_ids)} молекул")


# --------------------------------------------------
# 4. Создаём submission.csv
# --------------------------------------------------

# sample задаёт требуемый формат и порядок molecule_id.
submission_rows = []

for molecule_id in sample["molecule_id"]:
    ranked = rankings.get(molecule_id)

    if ranked is None or ranked.empty:
        # Не оставляем пустые поля: резервный поиск по всей базе.
        group = test[test["molecule_id"] == molecule_id]
        qmass = neutral_mass(
            group.iloc[0]["precursor_mz"],
            group.iloc[0]["adduct"],
        )

        tol = qmass * MASS_TOL_PPM * 1e-6
        left = np.searchsorted(ref_mass, qmass - tol)
        right = np.searchsorted(ref_mass, qmass + tol)

        fallback = (
            ref.iloc[left:right]
            .drop_duplicates("inchikey14")
            .head(TOP_N)
        )

        smiles_list = fallback["normalized_smiles"].tolist()
    else:
        smiles_list = ranked["smiles"].drop_duplicates().tolist()

    # Формат конкурса: несколько SMILES через точку с запятой.
    smiles_list = [s for s in smiles_list if isinstance(s, str) and s.strip()]
    smiles_list = smiles_list[:TOP_N]

    if not smiles_list:
        raise ValueError(
            f"Нет кандидатов для {molecule_id}; "
            "нужно проверить справочник и массы."
        )

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(smiles_list),
    })

submission = pd.DataFrame(submission_rows)

output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)

# --------------------------------------------------
# 5. Проверяем формат файла
# --------------------------------------------------

assert list(submission.columns) == ["molecule_id", "smiles"]
assert len(submission) == sample["molecule_id"].nunique()
assert submission["molecule_id"].is_unique
assert submission["smiles"].notna().all()
assert submission["smiles"].str.len().gt(0).all()
assert submission["smiles"].apply(
    lambda x: len(x.split(";")) <= TOP_N
).all()

print("\nФайл создан:", output_path)
print("Количество строк:", len(submission))
print("Максимум кандидатов:", submission["smiles"].str.count(";").max() + 1)
display(submission.head())

Эталонных спектров: 821960
Уникальных структур: 179693
Обработано 25/400 молекул
Обработано 50/400 молекул
Обработано 75/400 молекул
Обработано 100/400 молекул
Обработано 125/400 молекул
Обработано 150/400 молекул
Обработано 175/400 молекул
Обработано 200/400 молекул
Обработано 225/400 молекул
Обработано 250/400 молекул
Обработано 275/400 молекул
Обработано 300/400 молекул
Обработано 325/400 молекул
Обработано 350/400 молекул
Обработано 375/400 молекул
Обработано 400/400 молекул

Файл создан: /kaggle/working/submission.csv
Количество строк: 400
Максимум кандидатов: 25


,molecule_id,smiles
0,m_005e53,Cc1ccc(C(=O)NCC(=O)Nc2ccc(Br)c(C)c2)cc1C;CN(Cc...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;COc1...
2,m_00b5aa,COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;COC1CN(Cc2nnn...
3,m_01e922,Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;COc1ccc(CNC(...
4,m_0259d4,O=C(NCCCn1ccnc1)NCCc1coc(-c2ccccc2)n1;COC1CC(c...


In [14]:

path = "/kaggle/working/submission.csv"

print("Файл существует:", os.path.exists(path))
print("Содержимое /kaggle/working:")
print(os.listdir("/kaggle/working"))

if os.path.exists(path):
    df = pd.read_csv(path)
    print("\nРазмер:", df.shape)
    print("Колонки:", df.columns.tolist())
    print("Пустые значения:\n", df.isna().sum())
    display(df.head())

Файл существует: True
Содержимое /kaggle/working:
['__notebook__.ipynb', 'submission.csv']

Размер: (400, 2)
Колонки: ['molecule_id', 'smiles']
Пустые значения:
 molecule_id    0
smiles         0
dtype: int64


,molecule_id,smiles
0,m_005e53,Cc1ccc(C(=O)NCC(=O)Nc2ccc(Br)c(C)c2)cc1C;CN(Cc...
1,m_006153,COc1ccc(N2CC(C(=O)Nc3nncn3C3CC3)CC2=O)cc1;COc1...
2,m_00b5aa,COc1ccnc(CN2CCOCC3(CCCC3)C2)c1OC;COC1CN(Cc2nnn...
3,m_01e922,Cc1ccc(C(=O)OCC2CCCCC2)c(=O)[nH]1;COc1ccc(CNC(...
4,m_0259d4,O=C(NCCCn1ccnc1)NCCc1coc(-c2ccccc2)n1;COC1CC(c...


In [15]:
submission_path = "/kaggle/working/submission.csv"

assert os.path.isfile(submission_path), "submission.csv отсутствует"

print("Готовый файл:", submission_path)
print("Размер, байт:", os.path.getsize(submission_path))

Готовый файл: /kaggle/working/submission.csv
Размер, байт: 365797
